**Load  dummy data in patients table**


In [0]:
from pyspark.sql.functions import (
    col, lit, rand, when, expr, current_timestamp, to_date
)

# -----------------------------------
# CONFIG
# -----------------------------------
NUM_RECORDS = 20_000_000
TARGET_TABLE = "benefit_modernization_dev.bronze.join_patients"

# -----------------------------------
# BASE RANGE (DISTRIBUTED)
# -----------------------------------
df = spark.range(1, NUM_RECORDS + 1) \
          .withColumnRenamed("id", "patient_id")

# -----------------------------------
# DUMMY DATA GENERATION
# -----------------------------------
df_patients = (
    df
    .withColumn(
        "first_name",
        expr("""
            element_at(
              array('John','Jane','Michael','Sarah','David','Emily','Chris','Anna','Daniel','Sophia'),
              cast(rand()*10 as int) + 1
            )
        """)
    )
    .withColumn(
        "last_name",
        expr("""
            element_at(
              array('Smith','Johnson','Brown','Taylor','Anderson','Thomas','Jackson','White','Harris','Martin'),
              cast(rand()*10 as int) + 1
            )
        """)
    )
    .withColumn(
        "dob",
        expr("date_sub(current_date(), cast(rand()*20000 as int))")  # ~0–55 yrs
    )
    .withColumn(
        "gender",
        when(rand() < 0.49, lit("Male"))
        .when(rand() < 0.98, lit("Female"))
        .otherwise(lit("Other"))
    )
    .withColumn(
        "phone",
        expr("concat('+1-', cast(100 + rand()*900 as int), '-', cast(100 + rand()*900 as int), '-', cast(1000 + rand()*9000 as int))")
    )
    .withColumn(
        "email",
        expr("concat(lower(first_name), '.', lower(last_name), patient_id, '@example.com')")
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
    .withColumn(
        "updated_at",
          current_timestamp()
    )

)
df_patients.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(TARGET_TABLE)


from pyspark.sql.functions import (
    col, lit, rand, when, expr, current_timestamp
)

# -----------------------------------
# CONFIG
# -----------------------------------
NUM_APPOINTMENTS = 30_000_000
MAX_PATIENT_ID = 30_000_000
TARGET_TABLE = "benefit_modernization_dev.bronze.join_appointments"

# -----------------------------------
# BASE DATASET
# -----------------------------------
df = (
    spark.range(1, NUM_APPOINTMENTS + 1)
         .withColumnRenamed("id", "appointment_id")
)

# -----------------------------------
# APPOINTMENT DATA
# -----------------------------------
df_appointments = (
    df
    .withColumn(
        "patient_id",
        ((col("appointment_id") % MAX_PATIENT_ID) + 1).cast("bigint")
    )
    .withColumn(
        "doctor_id",
        (rand() * 5000).cast("bigint") + 1   # 5k doctors
    )
    .withColumn(
        "appointment_type",
        expr("""
            element_at(
              array('CONSULTATION','FOLLOW_UP','SURGERY','LAB_TEST','THERAPY'),
              cast(rand()*5 as int) + 1
            )
        """)
    )
    .withColumn(
        "status",
        expr("""
            element_at(
              array('SCHEDULED','COMPLETED','CANCELLED','NO_SHOW'),
              cast(rand()*4 as int) + 1
            )
        """)
    )
    .withColumn(
        "scheduled_time",
        current_timestamp() 
    )
    .withColumn(
        "created_at",
       current_timestamp() 
    )
    .withColumn(
        "updated_at",
        current_timestamp() 
    )
)


df_appointments.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(TARGET_TABLE)


